# 08. Zero-Copy Views & Stride Tricks (5+ Years Interview Guide)
Exhaustive revision guide to low-level pointer stride manipulation, zero-RAM sliding windows with as_strided, and sliding_window_view on transaction amounts.

### Key 5-Year Interview Concepts Covered:
- **Low-Level Windowing with `as_strided()`**: Creating overlapping sliding windows over arrays with zero RAM allocation.
- **Custom Byte-Step Stride Calculation**: Deriving shapes `(N - window + 1, window)` and strides `(itemsize, itemsize)`.
- **Safe Windowing (`sliding_window_view`)**: NumPy 1.20+ safe stride utility.
- **2D Convolution Patch Extraction**: Extracting 3x3 image patches without copying.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### Low-Level Windowing with `as_strided()`
**Explanation**: Constructs a 5-transaction zero-copy sliding window matrix over the amounts vector with zero RAM overhead.

**Syntax**: `as_strided(amounts, shape=(n_win, win_sz), strides=(8, 8))`

In [2]:
from numpy.lib.stride_tricks import as_strided
win_sz = 5
n_win = len(amounts[:20]) - win_sz + 1
stride_b = amounts.itemsize
rolling_amounts = as_strided(amounts[:20], shape=(n_win, win_sz), strides=(stride_b, stride_b))
print('Zero-Copy Rolling Window Matrix (first 3 windows):\n', rolling_amounts[:3].round(2))
print('Shares memory buffer with original?:', rolling_amounts.base is not None)

Zero-Copy Rolling Window Matrix (first 3 windows):
 [[1216.33  324.99  136.66  124.21 1284.68]
 [ 324.99  136.66  124.21 1284.68 1263.7 ]
 [ 136.66  124.21 1284.68 1263.7   781.65]]
Shares memory buffer with original?: True


### Zero-RAM Moving Averages via Stride Tricks
**Explanation**: Calculates moving average transaction amounts instantaneously as `rolling_amounts.mean(axis=1)`.

**Syntax**: `rolling_amounts.mean(axis=1)`

In [3]:
moving_avg_spend = rolling_amounts.mean(axis=1)
print('Moving Average Spend (window=5):', moving_avg_spend[:5].round(2))

Moving Average Spend (window=5): [617.37 626.85 718.18 763.56 874.69]


### Safe Sliding Windows with `sliding_window_view()`
**Explanation**: NumPy 1.20+ safe stride wrapper avoiding segmentation faults.

**Syntax**: `sliding_window_view(amounts[:20], window_shape=5)`

In [4]:
from numpy.lib.stride_tricks import sliding_window_view
safe_rolling = sliding_window_view(amounts[:20], window_shape=5)
print('Safe Sliding Window Matrix (first 3):\n', safe_rolling[:3].round(2))

Safe Sliding Window Matrix (first 3):
 [[1216.33  324.99  136.66  124.21 1284.68]
 [ 324.99  136.66  124.21 1284.68 1263.7 ]
 [ 136.66  124.21 1284.68 1263.7   781.65]]


### 2D Patch Extraction on Matrix
**Explanation**: Extracts 3x3 patches from a 2D transaction correlation grid with zero memory duplication.

**Syntax**: `sliding_window_view(grid, (3, 3))`

In [5]:
mock_grid = amounts[:25].reshape(5, 5)
patches = sliding_window_view(mock_grid, window_shape=(3, 3))
print('Grid Shape:', mock_grid.shape, 'Patches 4D Shape:', patches.shape)
print('First 3x3 Patch:\n', patches[0, 0].round(2))

Grid Shape: (5, 5) Patches 4D Shape: (3, 3, 3, 3)
First 3x3 Patch:
 [[1216.33  324.99  136.66]
 [1263.7   781.65  363.54]
 [ 567.7  1805.16  609.83]]


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Zero-RAM Rolling Volatility in Transaction Flow
**Explanation**: Compute 10-period rolling standard deviation over transaction amounts using zero-copy sliding windows.

**Syntax**: `sliding_window_view(amounts, 10).std(axis=-1)`

In [6]:
rolling_vol = sliding_window_view(amounts[:100], 10).std(axis=-1)
print('10-Period Rolling Spending Volatility Head:', rolling_vol[:5].round(2))

10-Period Rolling Spending Volatility Head: [429.96 390.28 515.28 476.43 419.21]
